# SG-AOI — PP-YOLOE+ PCBA component detector training (Colab)

This notebook trains the **PP-YOLOE+ (CRN small)** component detector for the SG-AOI
pipeline on a **free Colab T4 GPU**, evaluates it, and exports the **CPU inference model**
(`model.pdmodel` + `model.pdiparams` + `infer_cfg.yml`) that the shop-floor app loads.

**What it does, end to end:**

1. Checks the GPU, clones PaddleDetection, installs PaddlePaddle-GPU.
2. You upload `pcba_dataset.zip` (dataset + training config) from your PC.
3. Auto-sets `num_classes` from your COCO annotations and cross-checks class
   names against `expected_components.json` (if included).
4. Downloads COCO-pretrained weights, runs a dataset sanity check.
5. Trains (with `--eval`), evaluates the best checkpoint, exports the
   inference model, and downloads it as `pcba_ppyoloe_export.zip`.

---

## Prerequisites — prepare `pcba_dataset.zip` on your PC

From the `pcba-aoi` project folder, zip these together:

```
pcba_dataset.zip
├── detection_dataset/                 ← from data/detection_dataset/
│   ├── images/
│   │   ├── train/*.jpg
│   │   └── val/*.jpg
│   └── annotations/
│       ├── instances_train.json       (COCO format)
│       └── instances_val.json         (COCO format)
├── ppyoloe_plus_custom.yml            ← from configs/detection/
└── expected_components.json           ← (optional) from data/golden/
```

> **Class-name rule:** the `categories[].name` values in your COCO annotations must
> **exactly match** the `class` strings in `data/golden/expected_components.json` —
> the rule engine matches detections by class name. The auto-check cell below will
> warn loudly on any mismatch.

## Runtime choice

**Runtime → Change runtime type → Hardware accelerator → T4 GPU** (free tier).

## Expected duration

Roughly **1–3 hours** on a free T4 for a typical dataset of 50–100 board images
at 80 epochs (fine-tuning from COCO weights).

## ⚠️ Free-tier session limits

Colab free sessions can disconnect after ~12 h or earlier when idle, and **everything
in the VM is wiped**. The optional Google-Drive copy in the export cell backs up
checkpoints — consider running the Drive mount early if your dataset is large.
If the session dies mid-training, re-run from the top and resume from the newest
checkpoint in `output/ppyoloe_plus_custom/` (`-r` flag on `tools/train.py`).


In [ ]:
# --- Step 1: make sure we actually got a GPU --------------------------------
!nvidia-smi

import subprocess
r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if r.returncode != 0 or "GPU" not in r.stdout:
    raise SystemExit(
        "\n\u274c No GPU detected.\n"
        "Go to Runtime \u2192 Change runtime type \u2192 Hardware accelerator \u2192 T4 GPU, "
        "then re-run this cell. Training on CPU would take days."
    )
print("\n\u2705 GPU found:", r.stdout.strip())


In [ ]:
# --- Step 2: clone PaddleDetection (pinned stable release) -------------------
# Pinned to release/2.9 — the latest stable branch (v2.9.0).
# To use a different release, change PD_BRANCH and re-run; check available
# branches at https://github.com/PaddlePaddle/PaddleDetection/branches
import os
PD_BRANCH = "release/2.9"

if not os.path.isdir("PaddleDetection"):
    !git clone --depth 1 -b release/2.9 https://github.com/PaddlePaddle/PaddleDetection.git
%cd PaddleDetection
!git log -1 --oneline
print("Branch pinned:", PD_BRANCH)


In [ ]:
# --- Step 3: install PaddlePaddle-GPU + PaddleDetection requirements ---------
# NOTE: if the default paddle wheel mismatches Colab's CUDA version (import
# error or 'CUDA driver version is insufficient'), pick the exact wheel for
# the CUDA version printed by nvidia-smi from:
#   https://www.paddlepaddle.org.cn/install/quick
# e.g. for CUDA 12.6:
#   !pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddlepaddle-gpu
!pip install -q -r requirements.txt

import paddle
print("paddle:", paddle.__version__)
print("device:", paddle.device.get_device())
assert "gpu" in paddle.device.get_device().lower(), (
    "PaddlePaddle does not see the GPU — install the CUDA-matching wheel "
    "from https://www.paddlepaddle.org.cn/install/quick and re-run."
)
print("\u2705 PaddlePaddle sees the GPU.")


## Upload your dataset

Run the cell below — a file picker will open. Select the `pcba_dataset.zip`
you prepared on your PC. The cell unzips it, **copies the dataset to
`PaddleDetection/dataset/pcba_components/`** and the config to
`PaddleDetection/configs/ppyoloe/ppyoloe_plus_custom.yml` (exactly where the
config's relative paths expect them), then lists the tree so you can verify.


In [ ]:
from google.colab import files
import zipfile, shutil, pathlib, subprocess

print("\u2b06\ufe0f  Choose pcba_dataset.zip from your PC in the file picker below...")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
assert zip_names, "\u274c No .zip uploaded — please upload pcba_dataset.zip"
zip_path = zip_names[0]
print(f"Received: {zip_path} ({len(uploaded[zip_path]) / 1e6:.1f} MB)")

STAGING = pathlib.Path("/content/pcba_upload")
if STAGING.exists():
    shutil.rmtree(STAGING)
STAGING.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(STAGING)

# --- locate the dataset root (the dir that holds annotations/instances_train.json) ---
train_jsons = list(STAGING.rglob("instances_train.json"))
assert train_jsons, ("\u274c instances_train.json not found in the zip — "
                     "did you zip the contents of data/detection_dataset?")
dataset_src = train_jsons[0].parent.parent
print("Dataset root found at:", dataset_src.relative_to(STAGING))

dst = pathlib.Path("dataset/pcba_components")
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(dataset_src, dst)

# --- copy the training config into configs/ppyoloe/ ---
ymls = list(STAGING.rglob("ppyoloe_plus_custom.yml"))
assert ymls, "\u274c ppyoloe_plus_custom.yml not found in the zip"
shutil.copy(ymls[0], "configs/ppyoloe/ppyoloe_plus_custom.yml")
print("\u2705 Config copied to configs/ppyoloe/ppyoloe_plus_custom.yml")

# --- optional: expected_components.json for the class-name cross-check ---
exp = list(STAGING.rglob("expected_components.json"))
if exp:
    shutil.copy(exp[0], "/content/expected_components.json")
    print("\u2705 Found optional expected_components.json \u2192 will be used for the class-name check")
else:
    print("\u2139\ufe0f  No expected_components.json in the zip (optional) — cross-check will be skipped")

# --- verify the tree ---
print("\n--- dataset/pcba_components ---")
print(subprocess.run(["find", "dataset/pcba_components", "-maxdepth", "2"],
                     capture_output=True, text=True).stdout)
n_train = len(list((dst / "images/train").glob("*")))
n_val = len(list((dst / "images/val").glob("*")))
assert n_train > 0 and n_val > 0, "\u274c No images found under images/train or images/val"
print(f"train images: {n_train} | val images: {n_val}")


In [ ]:
# --- Step 4: auto-set num_classes from the COCO annotations ------------------
import json, re, pathlib

cfg_path = pathlib.Path("configs/ppyoloe/ppyoloe_plus_custom.yml")
train_anno = json.load(open("dataset/pcba_components/annotations/instances_train.json"))

categories = train_anno["categories"]
class_names = [c["name"] for c in sorted(categories, key=lambda c: c["id"])]
n_classes = len(categories)
print(f"Detected {n_classes} classes in instances_train.json:")
for i, name in enumerate(class_names):
    print(f"  [{i}] {name}")

# Rewrite the num_classes line with plain Python (no sed needed)
cfg_text = cfg_path.read_text()
new_text, n = re.subn(
    r"^num_classes:\s*\d+.*$",
    f"num_classes: {n_classes}   # auto-set from instances_train.json",
    cfg_text, count=1, flags=re.M)
assert n == 1, "Could not find a 'num_classes:' line to rewrite"
cfg_path.write_text(new_text)
print(f"\n\u2705 num_classes set to {n_classes} in {cfg_path}")

# Optional cross-check against data/golden/expected_components.json
exp_path = pathlib.Path("/content/expected_components.json")
if exp_path.exists():
    golden = json.load(open(exp_path))
    golden_classes = sorted({c["class"] for c in golden["components"]})
    missing_in_ds = set(golden_classes) - set(class_names)
    extra_in_ds = set(class_names) - set(golden_classes)
    if missing_in_ds or extra_in_ds:
        print("\n\u26a0\ufe0f\u26a0\ufe0f  CLASS-NAME MISMATCH vs expected_components.json  \u26a0\ufe0f\u26a0\ufe0f")
        if missing_in_ds:
            print("   in golden but NOT in dataset:", sorted(missing_in_ds))
        if extra_in_ds:
            print("   in dataset but NOT in golden:", sorted(extra_in_ds))
        print("   The rule engine matches detections by class name — fix the")
        print("   annotations or the golden file BEFORE training!")
    else:
        print("\n\u2705 Class names match expected_components.json exactly")
else:
    print("\n\u2139\ufe0f  expected_components.json not uploaded — cross-check skipped.")
    print("   Verify manually that class names match data/golden/expected_components.json!")


In [ ]:
# --- Step 5: download COCO-pretrained weights --------------------------------
import pathlib

url = "https://paddledet.bj.bcebos.com/models/ppyoloe_plus_crn_s_80e_coco.pdparams"
weights_dir = pathlib.Path("pretrained")
weights_dir.mkdir(exist_ok=True)
out = weights_dir / "ppyoloe_plus_crn_s_80e_coco.pdparams"

if not out.exists():
    !wget -q -O pretrained/ppyoloe_plus_crn_s_80e_coco.pdparams https://paddledet.bj.bcebos.com/models/ppyoloe_plus_crn_s_80e_coco.pdparams

size = out.stat().st_size
assert size > 0, "\u274c Download failed or file is empty — check your network and the URL."
print(f"\u2705 {out}  ({size / 1e6:.1f} MB)")


## Optional: disable RandomFlip augmentation

The inherited base recipe (`configs/ppyoloe/_base_/ppyoloe_plus_reader.yml`)
includes a **RandomFlip** transform. For PCBA inspection, **board orientation is
semantically meaningful** — a horizontally mirrored board is a *wrong* board
(components would appear in impossible positions), so flip augmentation can
teach the detector things that never occur on the line.

**When to run the cell below:** if your boards always arrive in a fixed
orientation from a fixed camera (the normal AOI setup) → run it to disable
RandomFlip.

**When to skip it:** if your boards genuinely arrive mirrored/rotated in
production (e.g. loose manual handling with no fixture), leave RandomFlip on
so the model learns those variants.

Nothing here touches rotation by default — small translation/scale/brightness
jitter from the base recipe is safe and is kept.


In [ ]:
# OPTIONAL CELL — run only if you decided (markdown above) that flip
# augmentation is wrong for your line. It comments RandomFlip out of the
# inherited TrainReader transform list. Safe to re-run (idempotent).
import re, pathlib

reader_path = pathlib.Path("configs/ppyoloe/_base_/ppyoloe_plus_reader.yml")
text = reader_path.read_text()
new_text, n = re.subn(
    r"^(\s*- !RandomFlip.*)$",
    r"# \1   # disabled: board orientation is meaningful (SG-AOI)",
    text, flags=re.M)

if n == 0:
    print("\u2139\ufe0f  No active '!RandomFlip' line found (already disabled or recipe changed) — nothing to do.")
else:
    reader_path.write_text(new_text)
    print(f"\u2705 Commented out {n} RandomFlip transform(s) in {reader_path}:")
    for line in new_text.splitlines():
        if "RandomFlip" in line:
            print("   ", line.strip())


In [ ]:
# --- Step 6: dataset sanity check --------------------------------------------
import json, collections, pathlib

dst = pathlib.Path("dataset/pcba_components")
ok = True
for split in ["train", "val"]:
    imgs = list((dst / f"images/{split}").glob("*"))
    anno = json.load(open(dst / f"annotations/instances_{split}.json"))
    cat = {c["id"]: c["name"] for c in anno["categories"]}
    counts = collections.Counter(cat[a["category_id"]] for a in anno["annotations"])
    print(f"\n== {split} ==  {len(imgs)} images, {len(anno['annotations'])} boxes")
    for name, cnt in sorted(counts.items()):
        flag = ""
        if cnt < 100:
            flag = "  \u26a0\ufe0f < 100 instances — collect more examples of this class!"
            ok = False
        print(f"  {name:<28} {cnt:>6}{flag}")
    for name in cat.values():
        if name not in counts:
            print(f"  {name:<28}      0  \u274c declared class with no annotations!")
            ok = False

print("\n" + ("\u2705 All classes have >= 100 instances." if ok
      else "\u26a0\ufe0f  Some classes are thin — fine-tuning may still work, but expect weak mAP on them."))


## Train

The cell below runs the full fine-tune with periodic evaluation:

```
python tools/train.py -c configs/ppyoloe/ppyoloe_plus_custom.yml --eval
```

**Notes:**

- **Checkpoints** land in `output/ppyoloe_plus_custom/` (one snapshot every
  10 epochs per the config; `best_model.pdparams` tracks the best mAP).
- **Watch the logs** for `mAP` after each eval pass — on a clean dataset it
  typically climbs past 0.8 within the first ~20 epochs.
- **Duration:** ~1–3 h on a free T4 for 50–100 board images (80 epochs).
- **CUDA out of memory?** Edit `configs/ppyoloe/ppyoloe_plus_custom.yml`
  `train_batch_size: 8` → `4` (cell above with the yml editor, or the Colab
  file browser) and re-run.
- If the session died earlier, resume with
  `-r output/ppyoloe_plus_custom/<last_checkpoint>`.


In [ ]:
# ~1-3 h on a free T4. Checkpoints -> output/ppyoloe_plus_custom/
# OOM? set train_batch_size: 4 in configs/ppyoloe/ppyoloe_plus_custom.yml and re-run.
!python tools/train.py -c configs/ppyoloe/ppyoloe_plus_custom.yml --eval


## Evaluate the best checkpoint

**Quality gates before shipping this model to the line:**

- **Per-class mAP (AP50) ≳ 0.9** for every component class — small passives
  are the usual stragglers.
- **Missing-part recall ≈ 100%** is validated *later in the app* against
  `data/golden/expected_components.json` — a detector that misses a real
  component is worse than one that false-alarms.

If a class is below target, collect more examples of it (see the sanity-check
cell output) and re-train.


In [ ]:
!python tools/eval.py -c configs/ppyoloe/ppyoloe_plus_custom.yml -o weights=output/ppyoloe_plus_custom/best_model.pdparams


In [ ]:
# --- Export the CPU inference model ------------------------------------------
# (single-line command: IPython `!` does not support bash-style backslash continuations)
!python tools/export_model.py -c configs/ppyoloe/ppyoloe_plus_custom.yml -o weights=output/ppyoloe_plus_custom/best_model.pdparams output_dir=output_inference/pcba_ppyoloe

import pathlib
export_dir = pathlib.Path("output_inference/pcba_ppyoloe")
print("\nExported files:")
for p in sorted(export_dir.iterdir()):
    print(f"  {p.name:<20} {p.stat().st_size / 1e6:>8.2f} MB")

for required in ["model.pdmodel", "model.pdiparams", "infer_cfg.yml"]:
    assert (export_dir / required).exists(), f"\u274c missing {required} — export failed"
print("\n\u2705 Export complete: model.pdmodel + model.pdiparams + infer_cfg.yml")


In [ ]:
# --- Zip the export and download it to your PC --------------------------------
import shutil, pathlib
from google.colab import files

zip_path = shutil.make_archive(
    "pcba_ppyoloe_export", "zip",
    root_dir="output_inference", base_dir="pcba_ppyoloe")
print("Created:", zip_path, f"({pathlib.Path(zip_path).stat().st_size / 1e6:.1f} MB)")
files.download(zip_path)

# OPTIONAL checkpoint backup — Colab free sessions are wiped on disconnect.
# Uncomment to also save checkpoints + the export to your Google Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/sg_aoi_checkpoints
# !cp -r output/ppyoloe_plus_custom /content/drive/MyDrive/sg_aoi_checkpoints/
# !cp pcba_ppyoloe_export.zip /content/drive/MyDrive/sg_aoi_checkpoints/
# print("Backed up to Google Drive \u2192 MyDrive/sg_aoi_checkpoints/")


## Next steps on the shop-floor PC

1. **Unzip** `pcba_ppyoloe_export.zip` so that the three files land exactly at:

   ```
   pcba-aoi/models/detection/pcba_ppyoloe/
   ├── model.pdmodel
   ├── model.pdiparams
   └── infer_cfg.yml
   ```

2. **Enable live detection** in `pcba-aoi/configs/pipeline.yaml` — make sure
   the precomputed-detections fallback is off:

   ```yaml
   detection:
     model_dir: "models/detection/pcba_ppyoloe"
     precomputed_json: null
   ```

3. **Launch the app** from the `pcba-aoi` folder:

   ```
   streamlit run src/app.py
   ```

   → **System Check** must show the detection model **green**.

4. **Latency gate:** detection on CPU should be **< ~3 s per board**. If it is
   slower, drop the eval input size (see `infer_cfg.yml`) or reduce camera
   resolution — never skip the missing-part recall check to save time.
